In [1]:
import re
import pandas as pd
from pathlib import Path

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [ ]:
# =========================
# 1. 基本配置
# =========================
INPUT_FILE = "/Users/wangziyi/Desktop/news_mo_extracted.xlsx"   # 改成你的文件名
OUTPUT_CLEANED = "/Users/wangziyi/Desktop/news_mo_cleaned.xlsx"
OUTPUT_CLUSTER_READY = "/Users/wangziyi/Desktop/news_mo_cluster_ready.xlsx"
OUTPUT_VALUE_COUNTS = "/Users/wangziyi/Desktop/news_mo_value_counts.xlsx"

# 第一轮聚类
PRIMARY_CLUSTER_COLS = [
    "contact_primary",
    "trust_primary",
    "manipulation_primary",
    "operation_primary",
    "extraction_primary",
    "aftermath_primary",
    "psychological_vulnerability",
    "compliance_driver",
]


OPTIONAL_COLS = [
    "payment_inducement_action",
    "surface_scenario",
    "detail_quality",
    "crosslingual_confidence",
    "is_scam",
    "title",
    "content",
    "translated_content",
    "combined_text",
]

ALL_POSSIBLE_COLS = PRIMARY_CLUSTER_COLS + OPTIONAL_COLS


In [3]:
# =========================
# 2. 通用文本清洗函数
# =========================
def normalize_str(x):
    if pd.isna(x):
        return None
    x = str(x).strip()

    # 去掉全角空格和多余空格
    x = x.replace("\u3000", " ")
    x = re.sub(r"\s+", " ", x).strip()

    # 常见简繁体与错写统一
    replacements = {
        "虛": "虚",
        "貪": "贪",
        "僥": "侥",
        "倖": "幸",
        "獲": "获",
        "關": "关",
        "聯": "联",
        "帳": "账",
        "帳戶": "账户",
        "帳号": "账号",
        "詐": "诈",
        "騙": "骗",
        "續": "续",
        "匯": "汇",
        "轉": "转",
        "聯繫": "联系",
        "聯絡": "联络",
        "開": "开",
        "體": "体",
        "廣": "广",
        "選靶": "选靶",  # 占位，保持一致
        "targeting": "选靶",  # 避免中英混杂残留
        "recovery scam": "追损骗局",
        "victim揭穿退出": "受害者识破退出",
        "blackmail": "敲诈勒索",
        "黑mail勒索": "敲诈勒索",
        "虛假交易": "虚假交易",
        "貪婪僥倖": "贪婪侥幸",
        "獲取收益": "获取收益",
    }
    for old, new in replacements.items():
         x = x.replace(old, new)

    # 常见“未提及”脏值统一
    not_mentioned_patterns = {
        "X0", "X0_无", "X0_未使用", "X0_无次级", "未使用", "无",
        "CON0_未提及", "CONTACT0_未提及", "未提到", "没提到", "未说明"
    }
    if x in not_mentioned_patterns:
        return "未提及"

    # 空字符串
    if x == "":
        return None

    return x
def normalize_bool(x):
    if pd.isna(x):
        return None
    s = str(x).strip().lower()
    if s in {"true", "1", "yes", "y", "是", "真"}:
        return True
    if s in {"false", "0", "no", "n", "否", "假"}:
        return False
    return None


def normalize_quality(x):
    if pd.isna(x):
        return None
    s = normalize_str(x)
    if s in {"高", "high", "HIGH"}:
        return "高"
    if s in {"中", "medium", "mid", "MEDIUM"}:
        return "中"
    if s in {"低", "low", "LOW"}:
        return "低"
    return s
def normalize_confidence(x):
    if pd.isna(x):
        return None
    s = normalize_str(x)
    if s in {"高", "high", "HIGH"}:
        return "高"
    if s in {"中", "medium", "mid", "MEDIUM"}:
        return "中"
    if s in {"低", "low", "LOW"}:
        return "低"
    return s




In [4]:
# =========================
# 3. 读取数据
# =========================
df = pd.read_excel(INPUT_FILE)

# 列名统一
df.columns = [str(c).strip() for c in df.columns]

# 只清洗表里实际存在的列
existing_cols = [c for c in ALL_POSSIBLE_COLS if c in df.columns]

for col in existing_cols:
    if col == "is_scam":
        df[col] = df[col].apply(normalize_bool)
    elif col == "detail_quality":
        df[col] = df[col].apply(normalize_quality)
    elif col == "crosslingual_confidence":
        df[col] = df[col].apply(normalize_confidence)
    else:
        df[col] = df[col].apply(normalize_str)



In [5]:
# =========================
# 4. 特殊规则修正
# =========================

# 4.1 修“熟人切入 + 冒充熟人”里其实应是“借熟人背书”的情况
# 规则：如果 contact 是熟人切入，trust 是冒充熟人，
# 且文本中出现“介绍/媒人/工友/同事/朋友介绍对象”等字样，
# 更可能是“借熟人背书”
intro_keywords = [
    "介绍", "媒人", "工友", "同事", "朋友介绍", "介绍对象",
    "介绍女友", "介绍男友", "相亲", "牵线", "撮合", "介绍认识"
]

text_cols_for_judge = [c for c in ["title", "content", "translated_content", "combined_text"] if c in df.columns]

def merge_text(row):
    parts = []
    for c in text_cols_for_judge:
        val = row.get(c)
        if pd.notna(val) and val:
            parts.append(str(val))
    return " ".join(parts)

if {"contact_primary", "trust_primary"}.issubset(df.columns):
    combined_text_series = df.apply(merge_text, axis=1)

    mask_contact = df["contact_primary"].eq("熟人切入")
    mask_trust = df["trust_primary"].eq("冒充熟人")
    mask_intro = combined_text_series.str.contains("|".join(map(re.escape, intro_keywords)), case=False, na=False)

    df.loc[mask_contact & mask_trust & mask_intro, "trust_primary"] = "借熟人背书"


# 4.2 如果 payment_inducement_action 存在，且只出现“转账/汇款”等结果词，
# 但没有任何付款理由关键词，则保守改成“未提及”
inducement_keywords = [
    "验资", "验证", "核验", "安全账户", "监管账户",
    "保证金", "押金", "定金",
    "手续费", "服务费", "工本费", "清关费", "税费", "认证费",
    "入金", "充值", "投资", "加仓", "垫资", "补单",
    "解冻", "解封", "提现",
    "代付", "代购", "礼物", "礼品卡", "点卡"
]

transfer_only_keywords = [
    "转账", "汇款", "付款", "打款", "支付"
]

if "payment_inducement_action" in df.columns and len(text_cols_for_judge) > 0:
    combined_text_series = df.apply(merge_text, axis=1)

    mask_transfer_only = combined_text_series.str.contains("|".join(map(re.escape, transfer_only_keywords)), case=False, na=False)
    mask_no_inducement = ~combined_text_series.str.contains("|".join(map(re.escape, inducement_keywords)), case=False, na=False)

    # 只在当前字段为空/未提及/不明确时用这个规则，不强行覆盖已有明确标签
    current = df["payment_inducement_action"].fillna("")
    mask_current_weak = current.isin(["", "未提及", "不明确", None])

    df.loc[mask_transfer_only & mask_no_inducement & mask_current_weak, "payment_inducement_action"] = "未提及"


# 4.3 统一 surface_scenario 中可能出现的脏值
if "surface_scenario" in df.columns:
    scenario_map = {
        "虛假交易": "虚假交易",
        "虚假交易 ": "虚假交易",
        "戀愛交友": "恋爱交友",
        "技術支持": "技术支持",
    }
    df["surface_scenario"] = df["surface_scenario"].replace(scenario_map)


# 4.4 统一心理脆弱点与驱动字段中常见脏值
for col in ["psychological_vulnerability", "compliance_driver"]:
    if col in df.columns:
        fix_map = {
            "P3_貪婪僥倖": "P3_贪婪侥幸",
            "D3_獲取收益": "D3_获取收益",
        }
        df[col] = df[col].replace(fix_map)


# =========================
# 5. 生成聚类可用版本
# =========================
cluster_df = df.copy()

# 只保留诈骗样本
if "is_scam" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["is_scam"] == True].copy()

# 只保留中高质量
if "detail_quality" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["detail_quality"].isin(["高", "中"])].copy()

# 如果有跨语言置信度，排除低
if "crosslingual_confidence" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["crosslingual_confidence"].isin(["高", "中", None])].copy()

# 第一轮聚类字段缺失统一
existing_cluster_cols = [c for c in PRIMARY_CLUSTER_COLS if c in cluster_df.columns]
for col in existing_cluster_cols:
    cluster_df[col] = cluster_df[col].fillna("未提及")
    cluster_df[col] = cluster_df[col].replace({"": "未提及", "无": "未提及", "未使用": "未提及"})


# =========================
# 6. 导出结果
# =========================
df.to_excel(OUTPUT_CLEANED, index=False)
cluster_df.to_excel(OUTPUT_CLUSTER_READY, index=False)

# 导出字段值分布，方便你人工检查
with pd.ExcelWriter(OUTPUT_VALUE_COUNTS, engine="openpyxl") as writer:
    for col in [c for c in existing_cols if c not in ["title", "content", "translated_content", "combined_text"]]:
        vc = df[col].fillna("<<空值>>").value_counts(dropna=False).reset_index()
        vc.columns = [col, "count"]
        sheet_name = col[:31]  # Excel sheet 名限制 31 字符
        vc.to_excel(writer, sheet_name=sheet_name, index=False)

print("清洗完成。")
print(f"完整清洗版：{OUTPUT_CLEANED}")
print(f"聚类可用版：{OUTPUT_CLUSTER_READY}")
print(f"字段值检查表：{OUTPUT_VALUE_COUNTS}")
print("第一轮建议聚类字段：", existing_cluster_cols)

清洗完成。
完整清洗版：/Users/wangziyi/Desktop/news_mo_cleaned.xlsx
聚类可用版：/Users/wangziyi/Desktop/news_mo_cluster_ready.xlsx
字段值检查表：/Users/wangziyi/Desktop/news_mo_value_counts.xlsx
第一轮建议聚类字段： ['contact_primary', 'trust_primary', 'manipulation_primary', 'operation_primary', 'extraction_primary', 'aftermath_primary', 'psychological_vulnerability', 'compliance_driver']


In [6]:
import zipfile, xml.etree.ElementTree as ET, pandas as pd, re

INPUT_FILE = "/Users/wangziyi/Desktop/news_mo_cleaned.xlsx"
OUTPUT_FILE = "/Users/wangziyi/Desktop/news_mo_cleaned_v2.xlsx"
OUTPUT_CLUSTER_READY = "/Users/wangziyi/Desktop/news_mo_cluster_ready_v2.xlsx"

# 读取 xlsx（不依赖 openpyxl/pandas 读引擎）
zf = zipfile.ZipFile(INPUT_FILE)
ns = {'a': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
sheet = ET.fromstring(zf.read('xl/worksheets/sheet1.xml'))
rows = sheet.find('a:sheetData', ns)

def col_to_idx(col):
    n = 0
    for ch in col:
        if ch.isalpha():
            n = n * 26 + (ord(ch.upper()) - 64)
    return n - 1

data = []
for row in rows:
    arr = {}
    for c in row:
        ref = c.attrib.get('r')
        idx = col_to_idx(''.join(filter(str.isalpha, ref)))
        v = c.find('a:v', ns)
        is_el = c.find('a:is', ns)
        txt = None
        if v is not None:
            txt = v.text
        elif is_el is not None:
            texts = [x.text or '' for x in is_el.iter() if x.tag.endswith('}t')]
            txt = ''.join(texts) if texts else None
        arr[idx] = txt
    data.append(arr)

max_col = max(max(r.keys()) for r in data)
table = [[arr.get(i) for i in range(max_col + 1)] for arr in data]
df = pd.DataFrame(table[1:], columns=table[0])

# ---------- 通用清洗 ----------
def norm(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s == "":
        return None
    return s

for col in df.columns:
    df[col] = df[col].apply(norm)

# 1) 所有 X0 系列统一成 未提及
x0_pattern = re.compile(r"^X0($|_)")

secondary_cols = [
    "prep_secondary", "contact_secondary", "trust_secondary",
    "manipulation_secondary", "operation_secondary",
    "extraction_secondary", "aftermath_secondary"
]

for col in secondary_cols:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: "未提及" if isinstance(x, str) and x0_pattern.match(x) else x
        )

# 2) secondary 里本来就有 *_未提及 也统一
for col in secondary_cols:
    if col in df.columns:
        df[col] = df[col].replace({
            "PREP0_未提及": "未提及",
            "CON0_未提及": "未提及",
            "TRU0_未提及": "未提及",
            "MAN0_未提及": "未提及",
            "OPR0_未提及": "未提及",
            "EXT0_未提及": "未提及",
            "AFT0_未提及": "未提及",
        })

# 3) 串列污染修正
if "manipulation_secondary" in df.columns:
    df["manipulation_secondary"] = df["manipulation_secondary"].replace({
        "TRU6_小额返利验证": "未提及",
        "TRU0_未提及": "未提及",
        "TRU4_专业人设包装": "未提及",
        "MAN3_编造新由追骗": "未提及",
    })

if "aftermath_secondary" in df.columns:
    df["aftermath_secondary"] = df["aftermath_secondary"].replace({
        "MAN6_沉没成本追加": "未提及",
    })

# 4) prep_secondary 同义统一
if "prep_secondary" in df.columns:
    df["prep_secondary"] = df["prep_secondary"].replace({
        "PREP7_伪造官方文件模板": "PREP7_伪造凭证文件"
    })

# 5) operation_primary 错位修正
if "operation_primary" in df.columns:
    df["operation_primary"] = df["operation_primary"].replace({
        "OPR5_线下物理接触": "OPR0_未提及",
        "OPR5_线下现金交割": "OPR0_未提及",
    })

# 6) surface_scenario 简繁体和长尾合并
if "surface_scenario" in df.columns:
    df["surface_scenario"] = df["surface_scenario"].replace({
        "投資理財": "投资理财"
    })

    rare_map = {
        "旅游服务欺诈": "其他诈骗",
        "招生录取": "其他诈骗",
        "游戏类诈骗": "其他诈骗",
        "养老金冒领": "其他诈骗",
        "退费诈骗": "其他诈骗",
        "直播诈骗": "其他诈骗",
        "招生入学": "其他诈骗",
        "招商加盟": "其他诈骗",
        "法律咨询": "其他诈骗",
        "福利申领": "其他诈骗",
        "签证代办": "其他诈骗",
        "考试作弊服务": "其他诈骗",
        "签证移民": "其他诈骗",
        "网络赌博": "其他诈骗"
    }
    df["surface_scenario"] = df["surface_scenario"].replace(rare_map)

# 7) is_scam 转成布尔
if "is_scam" in df.columns:
    df["is_scam"] = df["is_scam"].map({"1": True, "0": False, 1: True, 0: False})

# 8) 生成 cluster ready
cluster_df = df.copy()
if "is_scam" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["is_scam"] == True].copy()
if "detail_quality" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["detail_quality"].isin(["高", "中"])].copy()

df.to_excel(OUTPUT_FILE, index=False)
cluster_df.to_excel(OUTPUT_CLUSTER_READY, index=False)

print("二次清洗完成：")
print(OUTPUT_FILE)
print(OUTPUT_CLUSTER_READY)

二次清洗完成：
/Users/wangziyi/Desktop/news_mo_cleaned_v2.xlsx
/Users/wangziyi/Desktop/news_mo_cluster_ready_v2.xlsx


In [7]:
import pandas as pd

INPUT_FILE = "/Users/wangziyi/Desktop/news_mo_cleaned_v2.xlsx"
OUTPUT_FILE = "/Users/wangziyi/Desktop/news_mo_cleaned_v3.xlsx"
OUTPUT_CLUSTER_READY = "/Users/wangziyi/Desktop/news_mo_cluster_ready_v3.xlsx"

# 读取
df = pd.read_excel(INPUT_FILE)
df.columns = [str(c).strip() for c in df.columns]

# -------------------------
# 1. is_scam 统一成布尔值
# -------------------------
if "is_scam" in df.columns:
    df["is_scam"] = (
        df["is_scam"]
        .astype(str)
        .str.strip()
        .map({
            "1": True,
            "0": False,
            "True": True,
            "False": False,
            "true": True,
            "false": False
        })
    )

# -------------------------
# 2. surface_scenario 统一“其他”
# -------------------------
if "surface_scenario" in df.columns:
    df["surface_scenario"] = (
        df["surface_scenario"]
        .astype(str)
        .str.strip()
        .replace({
            "其他": "其他诈骗"
        })
    )

# -------------------------
# 3. trust_primary 异常值修正
# -------------------------
# 这里先保守归并到“TRU4_专业人设包装”
# 如果你们后面人工看这1条发现更适合别的类，再手动改
if "trust_primary" in df.columns:
    df["trust_primary"] = (
        df["trust_primary"]
        .astype(str)
        .str.strip()
        .replace({
            "TRU1_人设身份伪造": "TRU4_专业人设包装"
        })
    )

# -------------------------
# 4. 生成聚类直接可用版
# -------------------------
cluster_df = df.copy()

# 只保留诈骗样本
if "is_scam" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["is_scam"] == True].copy()

# 只保留中高质量
if "detail_quality" in cluster_df.columns:
    cluster_df = cluster_df[cluster_df["detail_quality"].isin(["高", "中"])].copy()

# 如果以后你补了 crosslingual_confidence，这里可以打开
if "crosslingual_confidence" in cluster_df.columns:
    cluster_df = cluster_df[
        cluster_df["crosslingual_confidence"].isin(["高", "中"])
    ].copy()

# 第一轮聚类常用字段缺失统一
cluster_cols = [
    "contact_primary",
    "trust_primary",
    "manipulation_primary",
    "operation_primary",
    "extraction_primary",
    "aftermath_primary",
    "psychological_vulnerability",
    "compliance_driver",
]

existing_cluster_cols = [c for c in cluster_cols if c in cluster_df.columns]

for col in existing_cluster_cols:
    cluster_df[col] = (
        cluster_df[col]
        .fillna("未提及")
        .astype(str)
        .str.strip()
        .replace({
            "": "未提及",
            "无": "未提及",
            "未使用": "未提及"
        })
    )

# 导出
df.to_excel(OUTPUT_FILE, index=False)
cluster_df.to_excel(OUTPUT_CLUSTER_READY, index=False)

print("补丁完成。")
print(f"完整修正版：{OUTPUT_FILE}")
print(f"聚类可用版：{OUTPUT_CLUSTER_READY}")
print("第一轮建议聚类字段：")
print(existing_cluster_cols)

# 简单检查输出
print("\n[检查] is_scam 分布：")
if "is_scam" in df.columns:
    print(df["is_scam"].value_counts(dropna=False))

print("\n[检查] surface_scenario 分布前10：")
if "surface_scenario" in df.columns:
    print(df["surface_scenario"].value_counts(dropna=False).head(10))

print("\n[检查] trust_primary 分布前10：")
if "trust_primary" in df.columns:
    print(df["trust_primary"].value_counts(dropna=False).head(10))

补丁完成。
完整修正版：/Users/wangziyi/Desktop/news_mo_cleaned_v3.xlsx
聚类可用版：/Users/wangziyi/Desktop/news_mo_cluster_ready_v3.xlsx
第一轮建议聚类字段：
['contact_primary', 'trust_primary', 'manipulation_primary', 'operation_primary', 'extraction_primary', 'aftermath_primary', 'psychological_vulnerability', 'compliance_driver']

[检查] is_scam 分布：
is_scam
False    17334
True      1644
Name: count, dtype: int64

[检查] surface_scenario 分布前10：
surface_scenario
不适用     17334
冒充公权      412
虚假交易      338
投资理财      278
冒充客服      179
恋爱交友      125
招聘兼职       82
其他诈骗       63
冒充熟人       62
刷单返利       35
Name: count, dtype: int64

[检查] trust_primary 分布前10：
trust_primary
不适用            17334
TRU1_公权身份伪装      512
TRU2_机构品牌冒用      499
TRU4_专业人设包装      239
TRU3_熟人关系利用      178
TRU7_伪造凭证文件       66
TRU0_未提及          59
TRU5_群体氛围伪造       46
TRU6_小额返利验证       45
Name: count, dtype: int64


In [8]:
pip install pandas openpyxl kmodes

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
from pathlib import Path
from kmodes.kmodes import KModes

# =========================
# 1. 基本配置
# =========================
INPUT_FILE = "/Users/wangziyi/Desktop/news_mo_cleaned_v3.xlsx"
OUTPUT_DIR = Path("/Users/wangziyi/Desktop/cluster_outputs_v3")
OUTPUT_DIR.mkdir(exist_ok=True)

# 第一轮最推荐的 5 个主字段
CLUSTER_COLS = [
    "trust_primary",
    "manipulation_primary",
    "operation_primary",
    "extraction_primary",
    "psychological_vulnerability"
]

# 想试的 k
K_LIST = [6, 7, 8, 9]

# 每簇抽样数量
SAMPLE_N = 15

# 随机种子
RANDOM_STATE = 42

# =========================
# 2. 读取与筛选
# =========================
df = pd.read_excel(INPUT_FILE)
df.columns = [str(c).strip() for c in df.columns]

# 检查字段
needed_cols = CLUSTER_COLS + ["is_scam", "detail_quality"]
missing_cols = [c for c in needed_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"缺少字段: {missing_cols}")

# 只保留诈骗样本 + 中高质量
work_df = df[
    (df["is_scam"] == True) &
    (df["detail_quality"].isin(["高", "中"]))
].copy()

# 缺失统一
for col in CLUSTER_COLS:
    work_df[col] = (
        work_df[col]
        .fillna("未提及")
        .astype(str)
        .str.strip()
        .replace({
            "": "未提及",
            "无": "未提及",
            "未使用": "未提及"
        })
    )

print(f"用于聚类的样本数: {len(work_df)}")
print("聚类字段：", CLUSTER_COLS)

# =========================
# 3. 试跑多个 k
# =========================
summary_rows = []

for k in K_LIST:
    print(f"\n===== 开始跑 k={k} =====")

    km = KModes(
        n_clusters=k,
        init="Huang",
        n_init=10,
        verbose=0,
        random_state=RANDOM_STATE
    )

    labels = km.fit_predict(work_df[CLUSTER_COLS])

    result_df = work_df.copy()
    cluster_col = f"cluster_k{k}"
    result_df[cluster_col] = labels

    # 保存聚类结果
    result_path = OUTPUT_DIR / f"clustered_k{k}.xlsx"
    result_df.to_excel(result_path, index=False)

    # 记录对比信息
    summary_rows.append({
        "k": k,
        "cost": km.cost_,
        "n_samples": len(result_df)
    })

    # =========================
    # 4. 每个簇的画像
    # =========================
    profile_rows = []

    for c in sorted(result_df[cluster_col].unique()):
        sub = result_df[result_df[cluster_col] == c].copy()
        row = {
            "k": k,
            "cluster": c,
            "cluster_size": len(sub),
            "cluster_pct": round(len(sub) / len(result_df), 4)
        }

        for col in CLUSTER_COLS:
            vc = sub[col].value_counts(dropna=False)
            top1 = vc.index[0] if len(vc) >= 1 else None
            top2 = vc.index[1] if len(vc) >= 2 else None
            top3 = vc.index[2] if len(vc) >= 3 else None

            row[f"{col}_mode"] = top1
            row[f"{col}_top3"] = " | ".join(
                [str(x) for x in [top1, top2, top3] if x is not None]
            )

        # 自动拼一个“簇签名”，方便人工命名
        row["suggested_signature"] = (
            f"信任={row['trust_primary_mode']}；"
            f"操控={row['manipulation_primary_mode']}；"
            f"操作={row['operation_primary_mode']}；"
            f"提取={row['extraction_primary_mode']}；"
            f"心理={row['psychological_vulnerability_mode']}"
        )

        row["manual_cluster_name"] = ""
        profile_rows.append(row)

    profile_df = pd.DataFrame(profile_rows)
    profile_path = OUTPUT_DIR / f"cluster_profiles_k{k}.xlsx"
    profile_df.to_excel(profile_path, index=False)

    # =========================
    # 5. 每簇抽样 15 条
    # =========================
    sample_cols = [c for c in [
        cluster_col,
        "title",
        "surface_scenario",
        "trust_primary",
        "manipulation_primary",
        "operation_primary",
        "extraction_primary",
        "psychological_vulnerability",
        "compliance_driver",
        "behavior_chain_summary",
        "script_track_summary"
    ] if c in result_df.columns]

    sampled_df = (
        result_df.groupby(cluster_col, group_keys=False)
        .apply(lambda x: x.sample(min(SAMPLE_N, len(x)), random_state=RANDOM_STATE))
        .reset_index(drop=True)
    )

    sample_path = OUTPUT_DIR / f"cluster_samples_k{k}.xlsx"
    # 再保险过滤一次，确保 sample_cols 真正在 sampled_df 里
safe_sample_cols = [c for c in sample_cols if c in sampled_df.columns]

# 如果一个都没有，就直接导出全部列，避免报错
if len(safe_sample_cols) == 0:
    sampled_df.to_excel(sample_path, index=False)
else:
    sampled_df[safe_sample_cols].to_excel(sample_path, index=False)

    print(f"k={k} 完成")
    print(f"  聚类结果: {result_path}")
    print(f"  簇画像: {profile_path}")
    print(f"  抽样样本: {sample_path}")

# =========================
# 6. 不同 k 对比
# =========================
summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "k_comparison.xlsx"
summary_df.to_excel(summary_path, index=False)

print("\n===== 全部完成 =====")
print(summary_df)
print(f"\n已保存: {summary_path}")

print("\n建议：")
print("1. 先看 k=7 和 k=8")
print("2. 打开 cluster_profiles_k7.xlsx / cluster_profiles_k8.xlsx 看簇画像")
print("3. 再看 cluster_samples_k7.xlsx / cluster_samples_k8.xlsx 做人工命名")

用于聚类的样本数: 1596
聚类字段： ['trust_primary', 'manipulation_primary', 'operation_primary', 'extraction_primary', 'psychological_vulnerability']

===== 开始跑 k=6 =====

===== 开始跑 k=7 =====

===== 开始跑 k=8 =====

===== 开始跑 k=9 =====
k=9 完成
  聚类结果: /Users/wangziyi/Desktop/cluster_outputs_v3/clustered_k9.xlsx
  簇画像: /Users/wangziyi/Desktop/cluster_outputs_v3/cluster_profiles_k9.xlsx
  抽样样本: /Users/wangziyi/Desktop/cluster_outputs_v3/cluster_samples_k9.xlsx

===== 全部完成 =====
   k    cost  n_samples
0  6  2515.0       1596
1  7  2467.0       1596
2  8  2432.0       1596
3  9  2242.0       1596

已保存: /Users/wangziyi/Desktop/cluster_outputs_v3/k_comparison.xlsx

建议：
1. 先看 k=7 和 k=8
2. 打开 cluster_profiles_k7.xlsx / cluster_profiles_k8.xlsx 看簇画像
3. 再看 cluster_samples_k7.xlsx / cluster_samples_k8.xlsx 做人工命名


In [11]:
pip install pandas openpyxl scikit-learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 10.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import pairwise_distances
import hdbscan

df = pd.read_excel("/Users/wangziyi/Desktop/news_mo_cleaned_v3.xlsx")

cluster_cols = [
    "trust_primary",
    "manipulation_primary",
    "operation_primary",
    "extraction_primary",
    "psychological_vulnerability"
]

df = df[(df["is_scam"] == True) & (df["detail_quality"].isin(["高", "中"]))].copy()

for col in cluster_cols:
    df[col] = df[col].fillna("未提及").astype(str).str.strip()

records = df[cluster_cols].values.tolist()
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(records)

dist = pairwise_distances(X, metric="jaccard")

clusterer = hdbscan.HDBSCAN(
    metric="precomputed",
    min_cluster_size=40,
    min_samples=10
)

df["cluster_hdbscan"] = clusterer.fit_predict(dist)
print(df["cluster_hdbscan"].value_counts().sort_index())

cluster_hdbscan
-1    272
 0    149
 1    547
 2    163
 3     54
 4     54
 5     41
 6     50
 7    109
 8     96
 9     61
Name: count, dtype: int64


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [13]:
cluster_col = "cluster_hdbscan"
cluster_cols = [
    "trust_primary",
    "manipulation_primary",
    "operation_primary",
    "extraction_primary",
    "psychological_vulnerability"
]

profile_rows = []

for c in sorted([x for x in df[cluster_col].unique() if x != -1]):
    sub = df[df[cluster_col] == c].copy()
    row = {
        "cluster": c,
        "cluster_size": len(sub),
        "cluster_pct": round(len(sub) / len(df), 4)
    }
    for col in cluster_cols:
        vc = sub[col].value_counts(dropna=False)
        top1 = vc.index[0] if len(vc) >= 1 else None
        top2 = vc.index[1] if len(vc) >= 2 else None
        top3 = vc.index[2] if len(vc) >= 3 else None
        row[f"{col}_mode"] = top1
        row[f"{col}_top3"] = " | ".join([str(x) for x in [top1, top2, top3] if x is not None])
    profile_rows.append(row)

profile_df = pd.DataFrame(profile_rows)
profile_df.to_excel("/Users/wangziyi/Desktop/hdbscan_cluster_profiles.xlsx", index=False)
profile_df

,cluster,cluster_size,cluster_pct,trust_primary_mode,trust_primary_top3,manipulation_primary_mode,manipulation_primary_top3,operation_primary_mode,operation_primary_top3,extraction_primary_mode,extraction_primary_top3,psychological_vulnerability_mode,psychological_vulnerability_top3
0,0,149,0.0934,TRU2_机构品牌冒用,TRU2_机构品牌冒用 | TRU4_专业人设包装 | TRU7_伪造凭证文件,MAN2_高收益利诱,MAN2_高收益利诱 | MAN1_恐吓威胁施压 | MAN4_制造紧急时限,OPR0_未提及,OPR0_未提及 | OPR1_下载安装应用 | OPR4_注册账户加入群组,EXT2_第三方数字支付,EXT2_第三方数字支付 | EXT5_线下现金交割 | EXT1_银行转账,P3_贪婪侥幸,P3_贪婪侥幸 | P5_生存焦虑 | P4_情感孤独
1,1,547,0.3427,TRU2_机构品牌冒用,TRU2_机构品牌冒用 | TRU1_公权身份伪装 | TRU4_专业人设包装,MAN2_高收益利诱,MAN2_高收益利诱 | MAN1_恐吓威胁施压 | MAN4_制造紧急时限,OPR0_未提及,OPR0_未提及 | OPR3_点击链接填写信息 | OPR4_注册账户加入群组,EXT1_银行转账,EXT1_银行转账 | EXT2_第三方数字支付 | EXT0_未提及,P3_贪婪侥幸,P3_贪婪侥幸 | P1_权威服从 | P5_生存焦虑
2,2,163,0.1021,TRU2_机构品牌冒用,TRU2_机构品牌冒用 | TRU3_熟人关系利用 | TRU1_公权身份伪装,MAN4_制造紧急时限,MAN4_制造紧急时限 | MAN1_恐吓威胁施压 | MAN3_情感绑架操控,OPR0_未提及,OPR0_未提及 | OPR3_点击链接填写信息 | OPR6_上传证件人脸识别,EXT1_银行转账,EXT1_银行转账 | EXT0_未提及 | EXT2_第三方数字支付,P1_权威服从,P1_权威服从 | P4_情感孤独 | P3_贪婪侥幸
3,3,54,0.0338,TRU2_机构品牌冒用,TRU2_机构品牌冒用 | TRU3_熟人关系利用 | TRU0_未提及,MAN2_高收益利诱,MAN2_高收益利诱 | MAN4_制造紧急时限 | MAN1_恐吓威胁施压,OPR3_点击链接填写信息,OPR3_点击链接填写信息 | OPR0_未提及 | OPR1_下载安装应用,EXT1_银行转账,EXT1_银行转账 | EXT0_未提及 | EXT5_线下现金交割,P3_贪婪侥幸,P3_贪婪侥幸 | P2_损失恐惧 | P1_权威服从
4,4,54,0.0338,TRU2_机构品牌冒用,TRU2_机构品牌冒用 | TRU3_熟人关系利用 | TRU1_公权身份伪装,MAN4_制造紧急时限,MAN4_制造紧急时限 | MAN2_高收益利诱 | MAN1_恐吓威胁施压,OPR3_点击链接填写信息,OPR3_点击链接填写信息 | OPR0_未提及 | OPR2_共享屏幕远程控制,EXT6_账户权限接管,EXT6_账户权限接管 | EXT1_银行转账 | EXT3_加密货币转移,P1_权威服从,P1_权威服从 | P4_情感孤独 | P3_贪婪侥幸
5,5,41,0.0257,TRU2_机构品牌冒用,TRU2_机构品牌冒用 | TRU4_专业人设包装 | TRU7_伪造凭证文件,MAN2_高收益利诱,MAN2_高收益利诱,OPR4_注册账户加入群组,OPR4_注册账户加入群组 | OPR3_点击链接填写信息 | OPR1_下载安装应用,EXT1_银行转账,EXT1_银行转账 | EXT2_第三方数字支付 | EXT3_加密货币转移,P3_贪婪侥幸,P3_贪婪侥幸 | P4_情感孤独
6,6,50,0.0313,TRU1_公权身份伪装,TRU1_公权身份伪装,MAN1_恐吓威胁施压,MAN1_恐吓威胁施压,OPR0_未提及,OPR0_未提及,EXT5_线下现金交割,EXT5_线下现金交割,P1_权威服从,P1_权威服从
7,7,109,0.0683,TRU1_公权身份伪装,TRU1_公权身份伪装 | TRU4_专业人设包装 | TRU7_伪造凭证文件,MAN1_恐吓威胁施压,MAN1_恐吓威胁施压 | MAN2_高收益利诱,OPR1_下载安装应用,OPR1_下载安装应用 | OPR0_未提及 | OPR5_执行刷单任务,EXT1_银行转账,EXT1_银行转账 | EXT0_未提及 | EXT5_线下现金交割,P1_权威服从,P1_权威服从 | P3_贪婪侥幸
8,8,96,0.0602,TRU1_公权身份伪装,TRU1_公权身份伪装,MAN1_恐吓威胁施压,MAN1_恐吓威胁施压,OPR3_点击链接填写信息,OPR3_点击链接填写信息 | OPR2_共享屏幕远程控制,EXT1_银行转账,EXT1_银行转账,P1_权威服从,P1_权威服从
9,9,61,0.0382,TRU1_公权身份伪装,TRU1_公权身份伪装,MAN1_恐吓威胁施压,MAN1_恐吓威胁施压,OPR0_未提及,OPR0_未提及,EXT1_银行转账,EXT1_银行转账,P1_权威服从,P1_权威服从


In [14]:
noise_df = df[df["cluster_hdbscan"] == -1].copy()
noise_df[[
    "title",
    "trust_primary",
    "manipulation_primary",
    "operation_primary",
    "extraction_primary",
    "psychological_vulnerability"
]].sample(20, random_state=42)

,title,trust_primary,manipulation_primary,operation_primary,extraction_primary,psychological_vulnerability
341,Geldwäsche als Schlagwort: Warum Volksbank-Kun...,TRU2_机构品牌冒用,MAN1_恐吓威胁施压,OPR3_点击链接填写信息,EXT6_账户权限接管,P1_权威服从
5388,15 Milliarden US-Dollar Bitcoin gestohlen? XRP...,TRU2_机构品牌冒用,MAN2_高收益利诱,OPR4_注册账户加入群组,EXT3_加密货币转移,P3_贪婪侥幸
3501,South Cariboo senior hangs up on ‘Telus’ scam ...,TRU2_机构品牌冒用,MAN4_制造紧急时限,OPR3_点击链接填写信息,EXT1_银行转账,P1_权威服从
5566,Millionen Spotify-Kunden betroffen: Warnung vo...,TRU2_机构品牌冒用,MAN4_制造紧急时限,OPR3_点击链接填写信息,EXT1_银行转账,P2_损失恐惧
7216,Alleged Scammer Using Fake Rings Confronted in...,TRU0_未提及,MAN2_高收益利诱,OPR0_未提及,EXT5_线下现金交割,P3_贪婪侥幸
5778,Spunta un nuovo raggiro Ecco ‘Dante’ l’assicur...,TRU2_机构品牌冒用,MAN1_恐吓威胁施压,OPR5_执行刷单任务,EXT0_未提及,P1_权威服从
7434,拉萨城关拦阻一电信网络诈骗,TRU2_机构品牌冒用,MAN1_恐吓威胁施压,OPR6_上传证件人脸识别,EXT0_未提及,P5_生存焦虑
882,Βαρύτατες διώξεις στους 44 Ρομά για τη σπείρα ...,TRU2_机构品牌冒用,MAN1_恐吓威胁施压,OPR3_点击链接填写信息,EXT1_银行转账,P1_权威服从
6433,Что за мошенничество с проверкой качества воды...,TRU2_机构品牌冒用,MAN1_恐吓威胁施压,OPR3_点击链接填写信息,EXT1_银行转账,P1_权威服从
15164,4999元就能速成网红？,TRU4_专业人设包装,MAN2_高收益利诱,OPR0_未提及,EXT1_银行转账,P3_贪婪侥幸
